# D8 — mmBERT 도메인 사전학습 (경제 문장 500만, Colab)

**런타임 설정 (런타임 → 런타임 유형 변경)**
- GPU: **A100** 권장 (보이면 H100이 더 빠름). L4도 되지만 느림. **T4는 쓰지 않기**(bf16 미지원).
- A100 40GB면 충분하다(배치는 GPU 메모리에 맞춰 자동). RAM은 **표준**으로 충분(토큰을 디스크에서 바로 읽음).

**준비물 (내 드라이브 `ner_dapt/`)** — 최세진 PC에서 이미 올려 둠
- `econ_dapt_morph.txt.gz` : 경제 문장 500만을 형태소 경계에서 띄운 것(골드셋 510·Claude 표본 12,010 제외, 문장 글자만)
- `dapt_colab.py` : 학습 스크립트 (로컬 CPU 시험 통과: 저장·다시 읽기 확인·이어서 학습·토큰 캐시 재사용)

**순서**: 위에서부터 모두 실행. 연결이 끊기면 **1~4번 셀을 다시 실행**하면 마지막 체크포인트부터 이어서 학습한다.
마지막 6번 셀은 **가중치 저장 확인이 통과했을 때만** 런타임을 반납한다.

**입력은 형태소로 띄운 말뭉치(`econ_dapt_morph.txt.gz`)**: mmBERT 토크나이저는 원문에서 개체 끝을 뒤 글자와 한 토큰으로 묶어(예: `%,` `리가`) 개체 경계의 95.6%만 토큰과 맞는다. 형태소로 띄우면 99.99%. 그래서 **파인튜닝·추론도 반드시 `pretok="morph"`(ner_lib.morph_split)** 로 같은 모양을 넣는다.


In [ ]:
# 1. GPU 확인
import torch, subprocess
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (런타임 → 런타임 유형 변경)'
p = torch.cuda.get_device_properties(0)
print(p.name, f'{p.total_memory/2**30:.1f}GB', 'bf16', torch.cuda.is_bf16_supported(), 'capability', torch.cuda.get_device_capability(0))
if 'T4' in p.name: print('⚠️ T4는 bf16이 없어 느리고 메모리가 부족할 수 있습니다. A100으로 바꾸세요.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
# 2. 드라이브 연결과 파일 확인
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/ner_dapt'
for f in ['econ_dapt_morph.txt.gz', 'dapt_colab.py']:
    path = f'{ROOT}/{f}'
    print(f, os.path.exists(path), f'{os.path.getsize(path)/2**20:.0f}MB' if os.path.exists(path) else '← 없음')
assert os.path.getsize(f'{ROOT}/econ_dapt_morph.txt.gz') == 319_434_726, '말뭉치 파일이 아직 덜 올라왔습니다(드라이브 동기화 대기)'

In [ ]:
# 3. 패키지 (ModernBERT 구조는 transformers 4.48 이상 필요)
import transformers
from packaging import version
print('transformers', transformers.__version__)
if version.parse(transformers.__version__) < version.parse('4.48'):
    !pip -q install -U transformers accelerate
    print('업그레이드함 → 런타임 → 세션 다시 시작 후 1번 셀부터 다시 실행')

In [ ]:
# 4. 학습 (끊기면 1~4번을 다시 실행 → 이어서 학습)
NAME = 'dapt_mmbert_econ5m_morph'
!cd /content && python {ROOT}/dapt_colab.py --corpus {ROOT}/econ_dapt_morph.txt.gz --out_root {ROOT} --name {NAME}     --model jhu-clsp/mmBERT-base --max_len 512 --tokens_per_step 32768 --mlm 0.15 --lr 5e-5 --epochs 1

In [ ]:
# 5. 저장 확인
import json, os
rec = json.load(open(f'{ROOT}/results/{NAME}.json', encoding='utf-8'))
for k in ['gpu', 'micro_batch', 'grad_accum', 'tokens_per_step', 'steps', 'eval_loss_before', 'eval_loss_after', 'train_minutes_this_run', 'tensors_checked', 'token_cls_shared_tensors', 'verify_ok']:
    print(f'{k:>26}: {rec.get(k)}')
print(os.listdir(f'{ROOT}/models/{NAME}'))
VERIFY_OK = bool(rec.get('verify_ok')) and os.path.exists(f'{ROOT}/models/{NAME}/model.safetensors')
print('VERIFY_OK =', VERIFY_OK)

In [ ]:
# 6. 런타임 반납 (저장 확인이 통과했을 때만)
if VERIFY_OK:
    from google.colab import runtime
    print('가중치 저장 확인 완료 → 런타임 반납'); runtime.unassign()
else:
    print('저장 확인이 통과하지 않아 반납하지 않습니다. 5번 셀 출력과 학습 로그를 확인하세요.')